# ಪಾಠ 09 - ಮೆಟಾಕಾಗ್ನಿಷನ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಸೆಟಪ್

ಈ ನೋಟ್ಬುಕ್ Microsoft Agent Framework ಅನ್ನು ಬಳಸಿಕೊಂಡು Metacognition ವಿನ್ಯಾಸ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಆವಶ್ಯಕತೆಗಳು:**
- ಪರಿಸರ ಚರಗಳಿಂದ ಸಂರಚಿಸಿದ Azure OpenAI ನಿಯೋಜನೆ
- Azure CLI ಪ್ರಾಮಾಣೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಮೆಟಾಕಾಗ್ನಿಷನ್ ಎಂದರೆ ಏನು?

ಮೆಟಾಕಾಗ್ನಿಷನ್ ಎಂದರೆ **ಚಿಂತನೆಯ ಬಗ್ಗೆ ಚಿಂತಿಸುವುದು**. AI ಏಜೆಂಟ್‌ಗಳ ಸಂದರ್ಭದಲ್ಲಿ, ಇದರ ಅರ್ಥವೆಂದರೆ ಕೆಳಕಂಡ ಕಾರ್ಯಗಳನ್ನು ನಿರ್ವಹಿಸಬಲ್ಲ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸುವುದು:

- **ಸ್ವಯಂ-ಆಲೋಚನೆ** ತಮ್ಮ ಸ್ವಂತ ಫಲಿತಾಂಶಗಳು ಮತ್ತು ತರ್ಕಿಸುವ ಪ್ರಕ್ರಿಯೆಯ ಬಗ್ಗೆ
- **ದೋಷಗಳನ್ನು ಗುರುತಿಸುವುದು** ಮತ್ತು ಮೌನವಾಗಿ ವಿಫಲವಾಗದೆ ಸುಗಮವಾಗಿ ಪುನರುಸ್ಥಾಪನೆ ಹೊಂದುವುದು
- **ಮೌಲ್ಯಮಾಪನ** ತಮ್ಮ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಸಂಪೂರ್ಣವಾಗಿದೆಯೇ ಮತ್ತು ಸಹಾಯಕವಾಗಿದೆಯೇ ಎಂಬುದನ್ನು
- **ಸರಿಹೊಂದಿಸಿಕೊಳ್ಳುವುದು** ಆರಂಭಿಕ ವಿಧಾನ ಕೆಲಸಮಾಡದಾಗ ತಮ್ಮ ತಂತ್ರವನ್ನು ಬದಲಾಯಿಸುವುದು (ಉದಾ., ಬ್ಯಾಕ್ಅಪ್ ಸಿಸ್ಟಮ್‌ಗೆ ಹಿಂದಿರುಗುವುದು)

ಮೆಟಾಕಾಗ್ನಿಷನ್ ಏಜೆಂಟ್ ಕೇವಲ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸುವುದಲ್ಲ — ಅದು ತನ್ನ ಸ್ವಂತ ಕಾರ್ಯಕ್ಷಮತೆಯನ್ನು ಗಮನಿಸುತ್ತದೆ ಮತ್ತು ತಕ್ಷಣವೇ ಹೊಂದಿಕೊಳ್ಳುತ್ತದೆ.


## ಪ್ರಾಥಮಿಕ ಮತ್ತು ಬ್ಯಾಕಪ್ ಉಪಕರಣಗಳು

ಸಾಮಾನ್ಯ ಮೇಲುಚಿಂತನೆಯ ಮಾದರಿ ಒಂದು **ಫಾಲ್ಬ್ಯಾಕ್ ತಂತ್ರ** ಆಗಿದೆ. ಏಜೆಂಟ್ ಮೊದಲಿಗೆ ಪ್ರಾಥಮಿಕ ಉಪಕರಣವನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತದೆ; ಅದು ವಿಫಲವಾದರೆ (ಉದಾ., 404 ದೋಷ), ಏಜೆಂಟ್ ಆ ವಿಫಲತೆಯನ್ನು ಗುರುತಿಸಿ ಪಾರದರ್ಶಕವಾಗಿ ಬ್ಯಾಕಪ್ ಉಪಕರಣಕ್ಕೆ ಬದಲಾಗುತ್ತದೆ.

ಇದು ವಾಸ್ತವಿಕ ವ್ಯವಸ್ಥೆಗಳನ್ನಾಗಿ ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ, ಅಲ್ಲಿ ಪ್ರಾಥಮಿಕ ಸೇವೆಗಳು ಲಭ್ಯವಿಲ್ಲದಿರಬಹುದು ಮತ್ತು ಏಜೆಂಟ್‌ಗಳು ಪರ್ಯಾಯ ಮಾರ್ಗವನ್ನು ಆಯ್ಕೆಮಾಡುವ ಮುನ್ನ ಖಂಡಿತವಾಗಿ ಸಮಸ್ಯೆಯನ್ನು ಸ್ವಯಂ ನಿರ್ಣಯಿಸಬೇಕು.

ಕೆಳಗೆ ನಾವು ಎರಡು ವಿಮಾನ ಹುಡುಕುವ ಉಪಕರಣಗಳನ್ನು ವಿವರಿಸುತ್ತೇವೆ:
- **ಪ್ರಾಥಮಿಕ** — ಪ್ಯಾರಿಸ್, ಟೋಕಿಯೋ, ಮತ್ತು ಬಾರ್ಸಿಲೋನಾ ಅನ್ನು ಒಳಗೊಂಡಿದೆ
- **ಬ್ಯಾಕಪ್** — ಬೆರ್ಲಿನ್, ಸिड್ನಿ, ಮತ್ತು ನ್ಯೂಯಾರ್ಕ್ ಸಿಟಿಯನ್ನು ಒಳಗೊಂಡಿದೆ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ದೋಷ ಪುನರುದ್ದಾರ ಹೊಂದಿರುವ ಸ್ವ-ಪರಿಶೀಲನಾ ಏಜೆಂಟ್

ಕೆಳಗಿನ ಏಜೆಂಟ್‌ಗೆ ಪ್ರಾಥಮಿಕ ಫ್ಲೈಟ್ ವ್ಯವಸ್ಥೆಯನ್ನು ಮೊದಲು ಪ್ರಯತ್ನಿಸಲು, ವೈಫಲ್ಯಗಳನ್ನು ಗುರುತಿಸಲು ಮತ್ತು ಪಾರದರ್ಶಕವಾಗಿ ಬ್ಯಾಕ್‌ಅಪ್ ವ್ಯವಸ್ಥೆಗೆ ಹಿಂತಿರುಗಲು ಸೂಚಿಸಲಾಗಿದೆ. ಪ್ರತಿಯೊಂದು ಪ್ರತಿಕ್ರಿಯೆಯ ನಂತರ ಅದು ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಸ್ವಯಂ-ಮೌಲ್ಯಮಾಪನ ನಡೆಸುತ್ತದೆ, ಅದು ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಗೆ ಸಂಪೂರ್ಣವಾಗಿ ಉತ್ತರಿಸಿದೆಯೇ ಎಂದು.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ಸ್ವ-ಮೌಲ್ಯಮಾಪನ ಮಾದರಿ

ಮೆಟಾಕಾಗ್ನಿಷನ್‌ನ ಮತ್ತೊಂದು ಅಂಶವೆಂದರೆ **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಬೇರೆ ಏಜೆಂಟ್ (ಅಥವಾ ಎರಡನೇ ಓಟದಲ್ಲಿ ಅದೇ ಏಜೆಂಟ್) ಉತ್ತರವನ್ನು ಸಂಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಉಪಯುಕ್ತತೆಯ ದೃಷ್ಟಿಯಿಂದ ಪರಿಶೀಲಿಸುತ್ತದೆ.

ಕೆಳಗೆ ನಾವು ಮೂರು ಆಯಾಮಗಳಲ್ಲಿ ಪ್ರವಾಸ ಏಜೆಂಟ್‌ನ ಉತ್ತರಗಳನ್ನು ಅಂಕೆಮಾಡುವ `ResponseEvaluator` ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಅನ್ನು ಉಪಯೋಗಿಸಿ **ಮೆಟಾಕಾಗ್ನಿಟಿವ್ ಏಜೆಂಟ್ಗಳನ್ನು** ಹೇಗೆ ನಿರ್ಮಿಸುವುದು ಎಂಬುದನ್ನು ಕಲಿತಿರಿ:

- **ಸ್ವ-ಪರಿಶೀಲನೆ**: ತಮ್ಮದೇ ಯುಕ್ತಿವಿಚಾರಣೆಯನ್ನು ಗಮನಿಸಿ ಏನಾಯಿತು ಎಂಬುದನ್ನು ಪಾರದರ್ಶಕವಾಗಿ ತಿಳಿಸುವ ಏಜೆಂಟ್‌ಗಳು.
- **ಫಾಲ್ಬ್ಯಾಕ್‌ಗಳೊಂದಿಗೆ ದೋಷ ಪುನರುದ್ದಾರಣೆ**: ಪ್ರಾಥಮಿಕ + ಬ್ಯಾಕ್ಅಪ್ ಸಾಧನ ಮಾದರಿ, ಇಲ್ಲಿ ಏಜೆಂಟ್ ವೈಫಲ್ಯಗಳನ್ನು (ಉದಾ., 404 ದೋಷಗಳು) ಗುರುತಿಸಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಪರ್ಯಾಯ ಮೂಲವನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತದೆ.
- **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಪೂರ್ಣತೆ, ನಿಖರತೆ ಮತ್ತು ಸಹಾಯಕತೆಯ ದೃಷ್ಠಿಯಿಂದ ಪ್ರತಿಕ್ರಿಯೆಗಳಿಗೆ ಅಂಕಗಳು ನೀಡುವ ಪ್ರತ್ಯೇಕ ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್.

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚು ಸ್ಥಿರ, ಪಾರದರ್ಶಕ ಮತ್ತು ನಂಬಿಗಸ್ತವಾಗಿರಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ — ಉತ್ಪಾದನದ ನಿಯೋಜನೆಗಳಿಗಾಗಿ ಅವಶ್ಯಕವಾದ ಪ್ರಮುಖ ಗುಣಲಕ್ಷಣಗಳು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ನಿರಾಕರಣೆ:
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆ Co-op Translator (https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಒದಗಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಮರ್ಪಕತೆಗಳು ಇರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿರಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿನ ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಪ್ರಾಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ನಿರ್ಣಾಯಕ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೆ ಅಥವಾ ತಪ್ಪು ವಿವರಣೆಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
